# 70_test_flight — Stage A end-to-end smoke on Blackwell

Mirror of notebook 70 with reduced data + 1 epoch so we can verify
the FULL pipeline (HN mine → train → merge+push → catalog re-embed → eval)
runs cleanly before we commit ~10 GPU-hr to the production run.

**Knobs vs production (notebook 70):**
| Stage | Production (nb 70) | Test flight (this nb) |
|---|---|---|
| HN mining rows | full train (~12k) | 5000 (`--max-rows 5000`) |
| Training epochs | 2 | 1 |
| Triples cap | none | head -5000 |
| Hub repo | `recsys2026-bge-m3-music-v1` | `recsys2026-bge-m3-music-testflight` |
| Merged repo | `...-v1-merged` | `...-testflight-merged` |
| Eval rows | 500 | 100 |
| **Expected wallclock on Blackwell** | ~6-10 hr | ~80-130 min |

**What it proves:**
1. The custom PEFT-LoRA loop (`train_bi_encoder.py`) runs end-to-end without OOM/dtype issues.
2. `merge_and_unload()` + `push_to_hub()` succeed on a Hub repo we control.
3. The catalog re-embed writes to the DENSE_LOCAL-compatible path (`{cache}/dense_local/{safe_model}/{label}/track_embeddings.pkl`).
4. The offline eval can load that pickle, run inference, compute nDCG@20.

**Gate**: any positive nDCG@20 score (sanity, not quality). The full run targets >= 0.15.

**After this passes**: run notebook 70 for the production model.

In [ ]:
# 1) Setup — identical to nb 70.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
os.makedirs(src, exist_ok=True)
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' \
    'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' 'bm25s' 'torchao>=0.16' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' 'tensorboard'

In [ ]:
# 2) HN mine — 5000 train rows (vs ~12k in production).
# Force re-mine: delete stale JSONL so we don't accidentally train on a
# file from before the session_id fix (commit ed18ee2).
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
!rm -f experiments/cache/retrieval_v2/triples_bge_m3_testflight.jsonl
!cd /content/recsys2026 && python -u scripts/build_bi_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --output experiments/cache/retrieval_v2/triples_bge_m3_testflight.jsonl \
    --query-mode bge_m3_structured \
    --percpos-threshold 0.95 --pool-size 1000 --k-negs 15 \
    --max-rows 5000 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/testflight_hn_log.txt
!wc -l experiments/cache/retrieval_v2/triples_bge_m3_testflight.jsonl
# Sanity check: verify the new session_id field is present.
!head -1 experiments/cache/retrieval_v2/triples_bge_m3_testflight.jsonl | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('HAS session_id:', 'session_id' in r); print('keys:', sorted(r.keys()))"


In [ ]:
# 3) Train on the FULL mined test-flight data (whatever cell 2 produced).
# QUICK & DIRTY config to validate the session-disjoint val split fix:
#   - 2 epochs (was 5) — enough to see curve shape; ~60-90 min on Blackwell
#   - val/full_catalog_ndcg_at_20 every 50 steps — more honest-metric data points
# Compare these in-training val numbers to the dev nDCG in cell 6.
# IMPORTANT: verify cell 2 wrote ≥4000 triples AND keys include session_id.
!cd /content/recsys2026 && python -u scripts/train_bi_encoder.py \
    --triples experiments/cache/retrieval_v2/triples_bge_m3_testflight.jsonl \
    --output-dir /content/bge_m3_testflight \
    --hub-repo OrRim123/recsys2026-bge-m3-music-testflight \
    --results-dir /content/drive/MyDrive/recsys2026_retrieval_v2_cache/results/bge_m3_testflight \
    --epochs 2 \
    --per-device-batch-size 16 \
    --gradient-accumulation-steps 2 \
    --query-max-len 512 \
    --passage-max-len 192 \
    --logging-steps 25 \
    --val-every-n-steps 50 \
    --val-full-catalog-every-n-steps 50 \
    --checkpoint-every-n-epochs 1 \
    --merge --cleanup-after-push \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/testflight_train_log.txt


In [ ]:
# 3a) TensorBoard launcher — run this AFTER kicking off cell 3 (training).
# TB scans the live log dir and auto-refreshes every ~30 sec. Curves to watch:
#   val/full_catalog_ndcg_at_20  — honest signal; should climb across epochs
#   val/loss                       — should descend then plateau (or rise = overfit)
#   train/loss vs val/loss gap     — growing gap = overfit onset
%load_ext tensorboard
%tensorboard --logdir /content/bge_m3_testflight/runs --port=6006

In [ ]:
# 4) Re-embed the catalog using the merged test-flight model.
# IMPORTANT: writes to DENSE_LOCAL's expected path under a TESTFLIGHT label
# so it doesn't shadow the prod v1 catalog pickle.
import os, pickle, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from mcrs.retrieval_modules.bge_m3_format import format_track_text

HUB_REPO = 'OrRim123/recsys2026-bge-m3-music-testflight-merged'
EMBED_LABEL = 'bge-m3-music-testflight-merged'
CACHE_ROOT = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local'
safe_model = HUB_REPO.replace('/', '_')
out_dir = os.path.join(CACHE_ROOT, safe_model, EMBED_LABEL)
os.makedirs(out_dir, exist_ok=True)

model = SentenceTransformer(HUB_REPO, device='cuda')
tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
texts = [format_track_text(r.get('track_name','unknown'), r.get('artist_name'), r.get('album_name'), r.get('release_date'), r.get('tag_list')) for r in tm]
track_ids = [r['track_id'] for r in tm]
embs = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
embs = np.asarray(embs, dtype=np.float32)
out_path = os.path.join(out_dir, 'track_embeddings.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'track_ids': track_ids, 'track_mat': embs}, f)
print(f'wrote {len(track_ids)} embeddings → {out_path}')
# Symlink into experiments/cache/dense_local so DENSE_LOCAL's cache_dir lookup hits.
local_cache = '/content/recsys2026/experiments/cache/dense_local'
os.makedirs(local_cache, exist_ok=True)
local_link = os.path.join(local_cache, safe_model)
if not os.path.exists(local_link):
    os.symlink(os.path.join(CACHE_ROOT, safe_model), local_link)
print('symlink ready:', local_link)

In [ ]:
# 5) Offline dev eval — 1000 turns (was 100). Compares to in-training val/full_catalog_ndcg.
# If both numbers are close (within ~0.02), the session-disjoint fix is working AND
# train→dev distribution shift is small. If dev is much lower, that's a real gap signal.
import sys, math, os, pickle
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.retrieval_modules.bge_m3_format import format_query_text
from build_bi_encoder_training_data import _iter_conversation_turns

HUB_REPO = 'OrRim123/recsys2026-bge-m3-music-testflight-merged'
EMBED_LABEL = 'bge-m3-music-testflight-merged'
CACHE_ROOT = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local'
safe_model = HUB_REPO.replace('/', '_')
catalog_pkl = os.path.join(CACHE_ROOT, safe_model, EMBED_LABEL, 'track_embeddings.pkl')
with open(catalog_pkl, 'rb') as f:
    payload = pickle.load(f)
track_ids = payload['track_ids']
track_mat = payload['track_mat']
model = SentenceTransformer(HUB_REPO, device='cuda')

dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
rows = _iter_conversation_turns(dev)[:1000]
queries = [format_query_text(
    chat_history=r.get('chat_history') or [],
    current_user_query=r.get('current_user_query',''),
    user_profile=r.get('user_profile_raw'),
    conversation_goal=r.get('conversation_goal'),
    mode='bge_m3_structured',
) for r in rows]
q_emb = model.encode(queries, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
q_emb = np.asarray(q_emb, dtype=np.float32)
sims = q_emb @ track_mat.T
top20_idx = np.argpartition(-sims, kth=19, axis=1)[:, :20]
ri = np.arange(sims.shape[0])[:, None]
top20_sorted = top20_idx[ri, np.argsort(-sims[ri, top20_idx], axis=1)]
tid_to_idx = {tid: i for i, tid in enumerate(track_ids)}
ndcgs = []
n_gold_missing = 0
for i, r in enumerate(rows):
    gold = r['track_id']
    if gold not in tid_to_idx:
        n_gold_missing += 1
        ndcgs.append(0.0); continue
    gold_idx = tid_to_idx[gold]
    top20_tids = top20_sorted[i]
    if gold_idx in top20_tids:
        rank = list(top20_tids).index(gold_idx) + 1
        ndcgs.append(1.0 / math.log2(rank + 1))
    else:
        ndcgs.append(0.0)
mean_ndcg = float(sum(ndcgs) / len(ndcgs))
n_hits = sum(1 for n in ndcgs if n > 0)
print(f'DEV nDCG@20 ({len(rows)} rows): {mean_ndcg:.4f}')
print(f'  hit rate: {n_hits}/{len(rows)} ({100*n_hits/len(rows):.1f}%)')
print(f'  gold not in catalog: {n_gold_missing}/{len(rows)} ({100*n_gold_missing/len(rows):.1f}%)')
print('\nCompare to in-training val/full_catalog_ndcg_at_20 from cell 3 logs.')
print('  Close (Δ ≤ 0.02): leak fix working, train→dev distribution shift small.')
print('  Dev much lower:    leak fix working BUT real train/dev gap — investigate.')
